# TF-IDF Retrieval System

# Import Required Libraries

In [1]:
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create Knowledge Base

In [2]:
knowledgeBase = pd.DataFrame([
    {
        "chunk_id": "A",
        "product": "app",
        "text": "Refunds are allowed within 30 days of purchase.",
    },
    {
        "chunk_id": "B",
        "product": "app",
        "text": "Premium users can export reports as CSV files.",
    },
    {
        "chunk_id": "C",
        "product": "web",
        "text": "Web dashboard passwords must be at least 12 characters.",
    },
    {
        "chunk_id": "D",
        "product": "app",
        "text": "Refund requests require the original order ID.",
    },
])

display(knowledgeBase)

,chunk_id,product,text
0,A,app,Refunds are allowed within 30 days of purchase.
1,B,app,Premium users can export reports as CSV files.
2,C,web,Web dashboard passwords must be at least 12 ch...
3,D,app,Refund requests require the original order ID.


# Build Search Index

In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
documentMatrix = vectorizer.fit_transform(knowledgeBase["text"])

print("Indexed documents:", documentMatrix.shape[0])
print("Vocabulary size:", documentMatrix.shape[1])

Indexed documents: 4
Vocabulary size: 22


# Retrieve Relevant Context

In [ ]:
def retrieve(query, product=None, topK=2):
    queryVector = vectorizer.transform([query])
    similarityScores = cosine_similarity(
        queryVector,
        documentMatrix,
    ).ravel()

    results = knowledgeBase.copy()
    results["score"] = similarityScores

    # Metadata Filtering here
    if product:
        results = results[results["product"] == product]

    return (
        results
        .sort_values("score", ascending=False)
        .head(topK)
        .reset_index(drop=True)
    )

# Build Grounded Prompt

In [5]:
def buildGroundedPrompt(question, retrievedChunks, minScore=0.05):
    if retrievedChunks.empty:
        return "No relevant context was retrieved."

    if retrievedChunks["score"].max() < minScore:
        return "I do not know from the provided context."

    context = "\n".join(
        f"[{row.chunk_id}] {row.text}"
        for row in retrievedChunks.itertuples()
    )

    return (
        "Answer the question using only the context below. "
        "If the answer is not supported, say that you do not know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

# Run Example Query

In [6]:
question = "Can I get a refund and what do I need?"

retrievedChunks = retrieve(
    question,
    product="app",
    topK=2,
)

display(
    retrievedChunks[
        ["chunk_id", "product", "score", "text"]
    ]
)

groundedPrompt = buildGroundedPrompt(
    question,
    retrievedChunks,
)

print(groundedPrompt)

,chunk_id,product,score,text
0,D,app,0.408248,Refund requests require the original order ID.
1,A,app,0.000000,Refunds are allowed within 30 days of purchase.


Answer the question using only the context below. If the answer is not supported, say that you do not know.

Context:
[D] Refund requests require the original order ID.
[A] Refunds are allowed within 30 days of purchase.

Question: Can I get a refund and what do I need?


# Evaluate Retrieval Quality

In [7]:
def precisionAtK(retrievedIds, relevantIds, k):
    if k <= 0:
        raise ValueError("k must be greater than 0")

    topRetrieved = set(retrievedIds[:k])
    relevantSet = set(relevantIds)

    return len(topRetrieved & relevantSet) / k


retrievedIds = retrievedChunks["chunk_id"].tolist()
relevantIds = ["A", "D"]

precisionScore = precisionAtK(
    retrievedIds,
    relevantIds,
    k=2,
)

print(f"Precision@2: {precisionScore:.2f}")

Precision@2: 1.00
